<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week5/lab2/WPI_week5_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation Quantification And Analysis

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

**Clinical disclaimer:** This lab uses a public pet image dataset as a segmentation teaching dataset. Physical measurements are simplified classroom calculations, not clinical measurements.


## Learning Objectives

By the end of this lab, you should be able to:

- Load a trained segmentation checkpoint.
- Convert model probabilities into raw and post-processed masks.
- Compute area, simple volume estimates, and a severity-style index from masks.
- Explain how pixel spacing and slice thickness affect measurements.
- Distinguish 2D, 2.5D, and 3D segmentation concepts.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week5_lab2_responses_LastName_FirstName.docx`

Answer the Word response questions in 2-5 sentences each.


## Workflow

This lab follows a post-segmentation analysis pipeline:

`Trained Model -> Probability Map -> Threshold -> Post-Processing -> Mask -> Measurement -> Biomarker`

Run Week 5 Lab 1 first. This notebook expects the Lab 1 checkpoint file `best_unet.pt`.


## Setup

Run this setup cell first. The notebook installs required packages, clones the public course helper repo in Colab, and imports shared helpers.


In [ ]:
#@title Setup course environment
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scikit-image",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    repo_dir = Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])

sys.path.insert(0, str(repo_dir))

import numpy as np
import matplotlib.pyplot as plt

from skimage.morphology import binary_closing, disk, remove_small_objects

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from wpi_ai_bootcamp.data import load_oxford_pet_segmentation_subset
from wpi_ai_bootcamp.notebook import make_wpi_overlay, setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)


## Data Loading

Use the same dataset loader as Lab 1. The selected subset is deterministic so you can compare the same examples across labs.


In [ ]:
images, masks, metadata = load_oxford_pet_segmentation_subset(
    max_samples=96,
    image_size=128,
    binary_foreground=True,
    download=True,
    random_state=42,
)

source = metadata["source"]
print(source.name)
print(source.url)
print("images:", images.shape)
print("masks:", masks.shape)


## Hyperparameters

Only change values in this block when the notebook asks you to run a controlled comparison.


In [ ]:
# STUDENT-EDITABLE HYPERPARAMETERS
BASE_CHANNELS = 12
CHECKPOINT_PATH = "best_unet.pt"
THRESHOLD = 0.5
MORPH_RADIUS = 3
MIN_OBJECT = 30
PIXEL_SPACING_MM = 0.5
SLICE_THICKNESS_MM = 3.0
SHOW_EXAMPLE_INDEX = 0
INTENSITY_SHIFT = 0.0
NUM_SLICES_FOR_3D = 5

# TODO: For Part 5, change exactly one value above and record the result in your Word response.


## Part 1 — Load The Trained Model

Load the checkpoint generated by Week 5 Lab 1. If the file is missing, run Lab 1 first or upload the checkpoint into the current Colab session.


In [ ]:
class SegmentationArrayDataset(Dataset):
    def __init__(self, images, masks, intensity_shift=0.0):
        shifted = np.clip(images + intensity_shift, 0.0, 1.0)
        self.images = torch.tensor(shifted, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]


class TinyUNet(nn.Module):
    def __init__(self, base_channels=12):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        up = self.up(e2)
        return self.out(self.dec1(torch.cat([up, e1], dim=1)))

checkpoint_file = Path(CHECKPOINT_PATH)
if not checkpoint_file.exists():
    raise FileNotFoundError(
        "best_unet.pt was not found. Run Week 5 Lab 1 first, or upload the Lab 1 checkpoint into this Colab session."
    )

checkpoint = torch.load(checkpoint_file, map_location=DEVICE)
base_channels = int(checkpoint.get("base_channels", BASE_CHANNELS)) if isinstance(checkpoint, dict) else BASE_CHANNELS
state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
model = TinyUNet(base_channels).to(DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("Loaded checkpoint:", CHECKPOINT_PATH)


In [ ]:
dataset = SegmentationArrayDataset(images, masks, intensity_shift=INTENSITY_SHIFT)
image, gt_mask = dataset[SHOW_EXAMPLE_INDEX]

with torch.no_grad():
    logits = model(image.unsqueeze(0).to(DEVICE))
    probability = torch.sigmoid(logits).cpu().squeeze().numpy()

raw_mask = (probability > THRESHOLD).astype(np.float32)
print("Probability range:", float(probability.min()), float(probability.max()))
print("Raw mask foreground fraction:", float(raw_mask.mean()))


### Part 1 Assessment — Model Loading And Raw Output (20 pts)

Required notebook output: successful checkpoint load, probability range, and raw mask foreground fraction.

Word response Q1: What does the probability map represent before thresholding?

Grading criteria: checkpoint is loaded, raw output is produced, and the response correctly describes probability-to-mask conversion.


## Part 2 — Post-Processing

Apply morphological cleanup to reduce small noisy regions and smooth the raw mask.


In [ ]:
def postprocess_mask(mask_np):
    mask_bool = mask_np.astype(bool)
    mask_bool = binary_closing(mask_bool, disk(MORPH_RADIUS))
    mask_bool = remove_small_objects(mask_bool, min_size=MIN_OBJECT)
    return mask_bool.astype(np.float32)

processed_mask = postprocess_mask(raw_mask)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()
axes[0].imshow(image.squeeze(), cmap="gray")
axes[0].set_title("Image")
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title("Ground truth")
axes[2].imshow(raw_mask, cmap="gray")
axes[2].set_title("Raw mask")
axes[3].imshow(processed_mask, cmap="gray")
axes[3].set_title("Post-processed")
axes[4].imshow(make_wpi_overlay(image.squeeze().numpy(), raw_mask > 0.5))
axes[4].set_title("Raw overlay")
axes[5].imshow(make_wpi_overlay(image.squeeze().numpy(), processed_mask > 0.5))
axes[5].set_title("Processed overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


### Part 2 Assessment — Post-Processing (20 pts)

Required notebook output: raw mask, post-processed mask, and overlays.

Word response Q2: Why might post-processing be useful after thresholding a probability map?

Grading criteria: visual comparison is present and the response names a concrete post-processing effect.


## Part 3 — Quantification From Segmentation

Convert masks into simplified measurements. These calculations are for learning how segmentation affects downstream numbers.


In [ ]:
def area_mm2(mask, pixel_spacing_mm):
    return float(mask.sum() * pixel_spacing_mm * pixel_spacing_mm)


def volume_mm3(mask_stack, pixel_spacing_mm, slice_thickness_mm):
    return float(mask_stack.sum() * pixel_spacing_mm * pixel_spacing_mm * slice_thickness_mm)


def biomarker_index(mask):
    return float(mask.sum() / mask.size) if mask.size else 0.0

raw_area = area_mm2(raw_mask, PIXEL_SPACING_MM)
processed_area = area_mm2(processed_mask, PIXEL_SPACING_MM)
stack_processed = np.stack([processed_mask for _ in range(NUM_SLICES_FOR_3D)], axis=0)
processed_volume = volume_mm3(stack_processed, PIXEL_SPACING_MM, SLICE_THICKNESS_MM)
processed_biomarker = biomarker_index(processed_mask)

print("Raw area (mm^2):", round(raw_area, 2))
print("Processed area (mm^2):", round(processed_area, 2))
print("Processed volume estimate (mm^3):", round(processed_volume, 2))
print("Processed severity-style index:", round(processed_biomarker, 4))


In [ ]:
spacing_values = [0.3, 0.5, 1.0]
area_values = [area_mm2(processed_mask, spacing) for spacing in spacing_values]

plt.figure(figsize=(6, 4))
plt.plot(spacing_values, area_values, marker="o", color=WPI_COLORS["crimson"])
plt.xlabel("Pixel spacing (mm/pixel)")
plt.ylabel("Measured area (mm^2)")
plt.title("Effect Of Pixel Spacing On Area")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 3 Assessment — Measurement (20 pts)

Required notebook output: area, volume, biomarker value, and pixel-spacing plot.

Word response Q3: Why is pixel count alone not enough to report physical area?

Grading criteria: measurements are produced and the response connects pixel spacing to physical units.


## Part 4 — 2D, 2.5D, And 3D Concepts

This is a conceptual comparison only. The model is still a 2D model.


In [ ]:
area_2d = area_mm2(processed_mask, PIXEL_SPACING_MM)
neighbor_masks = [processed_mask, processed_mask, processed_mask]
mask_25d = (np.mean(neighbor_masks, axis=0) > 0.5).astype(np.float32)
area_25d = area_mm2(mask_25d, PIXEL_SPACING_MM)
volume_3d = volume_mm3(stack_processed, PIXEL_SPACING_MM, SLICE_THICKNESS_MM)

plt.figure(figsize=(7, 4))
plt.bar(["2D area", "2.5D area", "3D volume"], [area_2d, area_25d, volume_3d], color=[WPI_COLORS["gray"], WPI_COLORS["accent_green"], WPI_COLORS["crimson"]])
plt.title("Conceptual 2D vs 2.5D vs 3D Comparison")
plt.tight_layout()
plt.show()

thickness_values = [1.0, 3.0, 5.0]
volume_values = [volume_mm3(stack_processed, PIXEL_SPACING_MM, t) for t in thickness_values]
plt.figure(figsize=(6, 4))
plt.plot(thickness_values, volume_values, marker="o", color=WPI_COLORS["crimson"])
plt.xlabel("Slice thickness (mm)")
plt.ylabel("Estimated volume (mm^3)")
plt.title("Effect Of Slice Thickness On Volume")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 4 Assessment — Dimensional Thinking (20 pts)

Required notebook output: 2D/2.5D/3D comparison plot and slice-thickness plot.

Word response Q4: What is the key difference between 2D, 2.5D, and 3D segmentation concepts?

Grading criteria: plots are present and the response distinguishes slice-wise, contextual, and volumetric reasoning.


## Part 5 — One Controlled Comparison

Change exactly one parameter and compare the final measurement. Keep all other settings identical.

Recommended first comparison: change only `COMPARISON_PIXEL_SPACING_MM` below.


In [ ]:
# TODO: Change only this value for the required controlled comparison.
COMPARISON_PIXEL_SPACING_MM = 0.8

comparison_area = area_mm2(processed_mask, COMPARISON_PIXEL_SPACING_MM)
comparison_volume = volume_mm3(stack_processed, COMPARISON_PIXEL_SPACING_MM, SLICE_THICKNESS_MM)

print("Original pixel spacing:", PIXEL_SPACING_MM)
print("Comparison pixel spacing:", COMPARISON_PIXEL_SPACING_MM)
print("Original processed area:", round(processed_area, 2))
print("Comparison processed area:", round(comparison_area, 2))
print("Original processed volume:", round(processed_volume, 2))
print("Comparison processed volume:", round(comparison_volume, 2))


### Part 5 Assessment — Controlled Comparison (20 pts)

Required notebook output: original and comparison measurement values.

Word response Q5: Which single parameter did you change, and how did it affect area, volume, or biomarker interpretation?

Grading criteria: exactly one parameter changes, measurements are compared, and the interpretation is tied to the output.


## Optional Challenge

Try changing `INTENSITY_SHIFT` and rerun prediction. Explain whether the mask changes and why data shift can matter for segmentation models.


## Attribution

- Data: Oxford-IIIT Pet dataset, loaded with `torchvision.datasets.OxfordIIITPet` using segmentation trimaps.
- Dataset page: https://www.robots.ox.ac.uk/~vgg/data/pets/
- PyTorch loader documentation: https://docs.pytorch.org/vision/main/generated/torchvision.datasets.OxfordIIITPet.html
- Citation: O. M. Parkhi, A. Vedaldi, A. Zisserman, and C. V. Jawahar, *Cats and Dogs*, IEEE Conference on Computer Vision and Pattern Recognition, 2012.
- License note: The Oxford dataset page lists Creative Commons Attribution-ShareAlike 4.0 International for commercial/research download; image copyrights remain with original owners.
- Libraries: NumPy, Matplotlib, scikit-image, PyTorch, torchvision, and course helper code.
